In [1]:
import os
import random
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from rag_pipeline.query_engine import load_data, vectorstore, ask_question, is_supported_file

/Volumes/LaCie/Projects_portfolio/IntelliQA/venv/lib/python3.11/site-packages/tika/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)
/Volumes/LaCie/Projects_portfolio/IntelliQA/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1716.17it/s]


In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.testset import TestsetGenerator
from langchain_groq import ChatGroq

In [4]:
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
)

In [5]:
from ragas import EvaluationDataset, evaluate

In [6]:
from eval import prepare_testset_documents

testset_docs = prepare_testset_documents("eval_data")

Successfully parsed SouthernStarEnergyInc_20051202_SB-2A_EX-9_801890_EX-9_Affiliate Agreement.pdf
Successfully parsed GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10.1_Content License Agreement.pdf
Successfully parsed TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-10.1_Affiliate Agreement.pdf
Successfully parsed EbixInc_20010515_10-Q_EX-10.3_4049767_EX-10.3_Co-Branding Agreement.pdf
Successfully parsed AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf
Successfully parsed SalesforcecomInc_20171122_10-Q_EX-10.1_10961535_EX-10.1_Reseller Agreement.pdf
Successfully parsed IpassInc_20181203_8-K_EX-99.1_11445874_EX-99.1_Reseller Agreement.pdf
Successfully parsed TodosMedicalLtd_20190328_20-F_EX-4.10_11587157_EX-4.10_Marketing Agreement_ Reseller Agreement.pdf
Successfully parsed ArcGroupInc_20171211_8-K_EX-10.1_10976103_EX-10.1_Sponsorship Agreement.pdf
Successfully parsed hts_2024_revision_10_csv.csv
Successfully parsed Electric_Vehicle_Populat

In [7]:
from ragas.testset import TestsetGenerator

In [34]:
vectorstore_db = vectorstore(persist_directory="/Volumes/LaCie/Projects_portfolio/IntelliQA/chroma_db", documents=None)

Loading existing vector space


In [8]:
from ragas import RunConfig
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator_llm = LangchainLLMWrapper(ChatGroq(
                        model="llama-3.3-70b-versatile", 
                        temperature=0,
                        model_kwargs={"response_format": {"type": "json_object"}}
                        )
                    )
generator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

run_config = RunConfig(max_workers=3, timeout=180)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 453.22it/s]


In [25]:
docs = random.sample(testset_docs, 5)

In [10]:
testset = generator.generate_with_langchain_docs(
    docs, testset_size=30, run_config=run_config
)

Generating Samples: 100%|██████████| 32/32 [03:27<00:00,  6.49s/it]


In [11]:
df = testset.to_pandas()

In [12]:
print(df.columns.tolist()) 

['user_input', 'reference_contexts', 'reference', 'synthesizer_name']


In [13]:
print(df[["user_input", "reference"]].head(10).to_string())

                                                                                                                                                                 user_input                                                                                                                                                                                                                  reference
0                                                                                                                  What is the us-gaap:ContractualObligationDueInThirdYear?                                                                                                                                                    The us-gaap:ContractualObligationDueInThirdYear is $21,221 due in 2028.
1  What is the total amount of contractual obligations due in the third year, specifically for the year 2028, according to the us-gaap:ContractualObligationDueInThirdYear?                                               

In [19]:
from ragas import RunConfig
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama

generator_llm = LangchainLLMWrapper(
        ChatOllama(model="qwen2.5", temperature=0)
        )

generator_embeddings = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        )
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

# With real TPM headroom, concurrency now helps:
run_config = RunConfig(max_workers=8, timeout=180)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 531.68it/s]


In [26]:
print("docs:", len(docs))
print(docs[0].page_content[:300])

docs: 5
2015,2015,WI,Wisconsin,Behavioral Risk Factor Surveillance System,Physical Activity,Physical Activity - Behavior,Percent of adults who engage in no leisure-time physical activity,,Value,20.8,20.8,,,17.5,24.5,955,,45 - 54,,,,,"(44.393191174, -89.816370742)",PA,PA1,Q047,VALUE,55,Age (years),45 - 54,AG


In [20]:
ollama_testset = generator.generate_with_langchain_docs(
    docs, testset_size=30, run_config=run_config
    )

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]        Node 40d31c7a-ade7-429d-8c75-c1c91226e411 does not have a summary. Skipping filtering.
Node 4f54ece7-fa00-422f-8587-dbaae54a8e41 does not have a summary. Skipping filtering.
Node 952d0c20-d2be-4628-8097-d79c8bf10f3c does not have a summary. Skipping filtering.
Node d4807da3-3437-4244-91fc-29900aada771 does not have a summary. Skipping filtering.
Node a4cb67bf-baf5-4f80-9bb1-856ad8ef106a does not have a summary. Skipping filtering.
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/15 [00:00<?, ?it/s]unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<class 'NoneType'>'
unable to apply transformation: node.property('summary') must be a string, found '<clas

ValueError: No nodes that satisfied the given filer. Try changing the filter.

In [24]:
kg = generator.knowledge_graph
print("nodes:", len(kg.nodes))
have_summary = [n for n in kg.nodes if isinstance(n.properties.get("summary"), str)]
print("nodes with a string summary:", len(have_summary))

nodes: 5
nodes with a string summary: 0


In [ ]:
df.to_csv("eval_data/testset.csv", index=False)

In [31]:
testset.to_pandas().to_json("eval_data/golden_set_groq_v1.json", orient="records", indent=2)

In [39]:
from rag_pipeline.query_engine import ask_question

probe = ask_question("What is this document about?", vectorstore=vectorstore_db ,session_id="test_session")  # adjust args to your signature
print("type:", type(probe))
print(probe)

type: <class 'langchain_core.messages.base.TextAccessor'>
This document appears to be a financial or securities filing, specifically referencing various senior notes (bonds) issued by Charter Communications, Inc. It lists different notes with their respective interest rates and maturity dates, and provides links to exhibits and filings with the SEC (Securities and Exchange Commission). The context suggests that it is part of a larger document, possibly an annual report or a proxy statement, that is being filed with the SEC.


In [40]:
import pandas as pd
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    ResponseRelevancy,
)
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from rag_pipeline.query_engine import ask_question

session_id = "test_session"
golden_set = "eval_data/golden_set_groq_v1.json"

In [41]:
evaluator_llm = LangchainLLMWrapper(
                    ChatGroq(
                        model="llama3-70b-8192",
                        temperature=0,
                        model_kwargs={"response_format": "json_object"},
                    )
                )
evaluator_embeddings = LangchainEmbeddingsWrapper(
        HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 764.50it/s]


In [42]:
metrics = [
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
    ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
]

In [44]:
retriever = vectorstore_db.as_retriever(
                    search_kwargs={
                        "filter": {"session_id": session_id}
                        }
                    )

In [45]:
def get_rag_response(question: str, vectorstore_db, session_id: str)->dict:
    response = ask_question(question, vectorstore_db, session_id)
    answer = response['result']
    contexts = [doc.page_content for doc in response['source_documents']]
    return {"answer": answer, "contexts": contexts}